In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.preprocessing import OneHotEncoder,LabelEncoder

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)


In [ ]:
# Task 1: Write your code here:
#Read the dataset Q1_data.csv using read_csv()
import os
import pandas as pd
## Load CSV file
# Load the CSV file
#path
df = pd.read_csv("/kaggle/input/q1-ka-ai-2026/Q1_data.csv")


In [ ]:
# Task 2: Write your code here:
#2. Inspect the first few rows using `head()`
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
#Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('delivery time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df=df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
# Missing values
print("Missing values:")
print(df.isnull().sum())

In [ ]:
df.info()

In [ ]:
# Task 3: Write your code here:
# Fill categorical columns with mode
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
  df[col].fillna(df[col].mode()[0], inplace=True)



# Fill float columns with mean
for col in ['Courier_Experience_yrs', 'Delivery_Time']:
  df[col].fillna(df[col].mean(), inplace=True)


print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Task 4: Write your code here:
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Separate Features and Target
target_column = "Delivery_Time"

X = df.drop(target_column, axis=1)
y = df[target_column]
# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Encode Categorical Features
label_encoder = LabelEncoder()

for col in X.select_dtypes(include=["object"]).columns:
    X[col] = label_encoder.fit_transform(X[col])


# Scale features - fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)



In [ ]:
# # Task 5: Write your code here:

# # Task 1: Write your code here:
# print("Separating target variable and features...")
# X = df.drop('Delivery_Time', axis=1)
# y = df['Delivery_Time']


# from sklearn.preprocessing import OneHotEncoder,LabelEncoder
# categories=['Weather', 'Traffic_Level', 'Time_of_Day','Vehicle_Type']

# print("Applying One-Hot Encoding to feature DataFrame 'categories'...")

# # Encode Categorical Features
# label_encoder = LabelEncoder()

# for col in X.select_dtypes(include=["object"]).columns:
#     X[col] = LabelEncoder.fit_transform(X[col])


# # onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# # Feature Scaling
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)



# # onehot_encoder = OneHotEncoder[categories, sparse_output=False
# # onehotencoder = OneHotEncoder(categorical_features=df.categories)
# # onehotencoder = OneHotEncoder(df['Weather'])
# #Encoding the categorical data
# # onehotencoder=OneHotEncoder( categorical_features=df[categories]])

# # onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# # from sklearn.preprocessing import LabelEncoder

# # labelencoder_X = LabelEncoder()
# # df[:,0] = labelencoder_X.fit_transform(df[:,0])

# #we are dummy encoding as the machine learning algorithms will be
# #confused with the values like Spain > Germany > France
# # from sklearn.preprocessing import OneHotEncoder

# # onehotencoder = OneHotEncoder(categorical_features=[0])
# # X = onehotencoder.fit_transform(df).toarray()


# # print("Applying Label Encoding to target variable 'y'...")
# # label_encoder = LabelEncoder()
# # y_encoded = label_encoder.fit_transform(df[Distance_km])

# # print("Applying One-Hot Encoding to feature DataFrame 'categories'...")
# # onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# # X_encoded = pd.DataFrame(onehot_encoder.fit_transform(X), columns=onehot_encoder.get_feature_names_out(X.columns))




# # print("First 5 rows of X_encoded:")
# # display(X_encoded.head())
# # print("First 5 elements of y_encoded:")
# # print(y_encoded[:5])

In [ ]:
# Task 6: Write your code here:
import seaborn as sns

print("Target Distribution:")
print(df["Delivery_Time"].value_counts(normalize=True))
sns.countplot(df["Delivery_Time"])
plt.title("Target Distribution")
plt.show()



In [ ]:
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")


In [ ]:
from sklearn.model_selection import train_test_split, KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"MAE:  ${mae_scores.mean():,.2f}")


In [ ]:
feature_importance=['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs',]

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_importance,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('Distribution of Predictions( delivery time )')
plt.xlabel('predicted delivery time')
plt.ylabel('Count')
plt.show()

In [ ]:
!pip install catboost

In [ ]:
# Import models
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostRegressor
from sklearn.ensemble import VotingClassifier


In [ ]:
def one_hot_encode(labels, num_classes):
    """Convert integer labels to one-hot encoded format."""
    one_hot = np.zeros((len(labels), num_classes))
    one_hot[np.arange(len(labels)), labels] = 1
    return one_hot

In [ ]:
print("Applying Label Encoding to target variable 'y'...")
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [ ]:
def categorical_cross_entropy(y_true, y_pred):
    """Compute categorical cross-entropy loss."""
    epsilon = 1e-10 # Small value to prevent log(0)
    y_pred = np.clip(y_pred, epsilon, 1. - epsilon)
    loss = -np.sum(y_true * np.log(y_pred), axis=-1)
    return np.mean(loss)

In [ ]:
print("Initializing models and K-Fold cross-validation...")

# Determine the number of classes for one-hot encoding
num_classes = len(np.unique(y_encoded))

# 1. Initialize the machine learning models
models = {
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100),
     "CatBoost": CatBoostRegressor(verbose=0)
}


# 2. Create a KFold object
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Create an empty dictionary to store average losses
model_losses = {}

print("Starting K-Fold Cross-Validation for each model...")

# 4. For each model:
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    fold_MSE = [] # List to store loss from each fold

    for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
        # Split X_train and y_train into training and validation sets for the current fold
        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

        # Train the current model
        model.fit(X_train_fold, y_train_fold)

        # Generate predictions (probabilities) on the validation set
        y_pred_proba = model.predict_proba(X_val_fold)

        # Convert y_val_fold (true labels) into a one-hot encoded format
        y_val_one_hot = one_hot_encode(y_val_fold, num_classes)

        # Calculate mean_squared_error current folder
        MSE2 = mean_squared_error(y_val_one_hot, y_pred_proba)

        fold_MSE.append(MSE2)



    # Calculate the average loss for the model
    avg_fold_MSE = np.mean(fold_MSE)
    model_losses[model_name] = avg_fold_MSE

    # Print the average cross-validation loss for the current model
    print(f"{model_name} - Average fold_MSEs: {avg_fold_MSE:.4f}")

print("\nAll models trained and evaluated. Stored average losses:")
print(model_losses)